# Coordinator 패턴: 계획은 큰 model, 실행은 작은 model

## 소개

대부분의 agent 작업에는 매우 다른 두 일이 포함됩니다. 소량의 계획과 판단, 대량의 기계적인 읽기와 실행입니다. web 조사는 극단적인 사례이며 이 notebook의 예제로 사용합니다. 신뢰할 수 있는 출처로 사실 스무 개를 검증하려면 수십만 token의 web 페이지를 model로 읽어야 하며, frontier 요금에서는 이 읽기 비용이 지배적입니다.

coordinator 패턴은 두 작업을 나눕니다. frontier model이 조사를 계획하고 답을 종합하지만 원본 web 페이지에는 전혀 접근하지 않습니다. 저렴한 worker가 병렬의 독립적인 context window에서 모든 읽기를 수행하고 정제된 결과를 보고합니다. 이 notebook은 이 분리의 효과를 정직하게 측정합니다. 같은 tool과 같은 검증 기준을 가진 frontier agent 하나라는 현실적인 대안을 같은 질문에 실행하여 실제 청구액과 실제 경과 시간을 비교합니다. 작성자의 실행에서는 두 방식이 거의 같은 분량을 읽었으며, 팀의 input token 84-98%가 worker 요금으로 청구되었습니다.

**이 cookbook을 마치면 다음을 할 수 있습니다.**

- `multiagent` coordinator 필드로 두 model 팀 설정하기: frontier coordinator와 저렴한 검색 worker
- 세션 event stream으로 위임을 실시간으로 따라가기(`thread_created`, `thread_message_sent`, `thread_message_received`)
- 엄밀성이 같은 단독 frontier 대조 실행을 수행하고 실제 청구액 비교하기
- 타입이 지정된 thread별 누적 `usage`로 각 thread 사용량 측정하기

저렴한 model이 token을 많이 사용하는 부분을 수행할 수 있는 작업에는 같은 경제성이 적용됩니다. 문서 review, log 분석, codebase 전체 점검이 해당합니다.


![coordinator 패턴의 architecture: 사용자 질문이 자체 tool이 없는 frontier-model coordinator로 전달됩니다. 공원별 작업 지시를 병렬 small-model 검색 worker에 보내고 정제된 결과를 받습니다. web search와 page fetch를 통해 open web에 접근하는 것은 worker뿐입니다. coordinator는 최종 답을 종합합니다.](example_data/plan_big_execute_small/architecture_diagram.png)


## 사전 요구 사항

이 guide를 따르기 전에 다음을 준비하세요.

**필요한 지식:**
- Python 기초
- Managed Agents 기본 개념 — agent, environment, session, streaming event 루프에 대한 이해([`CMA_iterate_fix_failing_tests.ipynb`](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_iterate_fix_failing_tests.ipynb)에서 모두 소개합니다)

**필요한 도구:**
- Python 3.11 이상
- Managed Agents beta에 접근할 수 있는 Anthropic API key([여기서 발급](https://console.anthropic.com))

`multiagent` 필드를 처음 본다면 [`CMA_coordinate_specialist_team.ipynb`](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_coordinate_specialist_team.ipynb)에서 서로 다른 specialist 팀으로 소개합니다. 이 notebook은 가능한 가장 단순한 팀인 worker 타입 하나를 사용합니다. 여기서 핵심은 팀 설계가 아니라 비용 구조이기 때문입니다.

## 설정


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]

# frontier model은 계획하고 종합하며 저렴한 model은 web을 읽습니다.
COORDINATOR_MODEL = os.environ.get("COOKBOOK_COORDINATOR_MODEL", "claude-fable-5")
WORKER_MODEL = os.environ.get("COOKBOOK_WORKER_MODEL", "claude-sonnet-5")

client = anthropic.Anthropic()

## 1. 팀: 저렴한 reader, 비용이 높은 thinker

두 agent 정의로 전체 팀을 만듭니다.

**worker** — 문서 용어로는 coordinator가 roster에서 생성할 수 있는 subagent — 는 일반 agent입니다. model, `web_search` + `web_fetch`로 범위를 좁힌 toolset, system prompt로 구성됩니다. 각 worker instance는 자체 세션 thread에서 하나의 집중된 하위 질문을 조사하므로 읽는 거대한 web 페이지가 다른 누구의 context에도 들어가지 않습니다.

**coordinator**는 자체 tool이 없고 worker를 지정하는 `multiagent` roster만 있습니다. 그 필드 하나가 coordinator로 만듭니다. 서버는 자동으로 `create_agent`, `send_to_agent`, `wait_for_agents`, `list_agents`를 제공하고 worker도 같은 방식으로 `submit_result`와 `send_to_parent`를 받습니다. 이 tool을 직접 정의하지 않습니다.

이 관계에서 알아야 할 것은 두 가지입니다. 첫째, roster는 coordinator 생성 또는 update 시 snapshot됩니다. worker 정의를 변경하면 coordinator를 update하거나 다시 생성하세요. 둘째, 덜 분명한 점은 **coordinator가 roster agent에 관해 아무것도 볼 수 없다는 것**입니다. prompt, 이름, 설명 모두 볼 수 없습니다. `create_agent` tool은 agent 이름과 작업 문자열만 받습니다. coordinator가 worker에 대해 믿는 모든 내용은 자체 system prompt에서 나오므로 이 설명과 worker의 실제 prompt를 일치시키세요. 서버는 이를 강제하지 않습니다. (단일 agent roster에서는 요청한 이름이 무엇이든 하나의 worker로 연결됩니다. worker 타입이 여러 개면 coordinator prompt에 이름을 명시하세요.)


In [ ]:
worker = client.beta.agents.create(
    name="search-worker",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=WORKER_MODEL,
    # web tool 두 개를 제외하고 모두 비활성화합니다. worker의 업무는
    # 읽기이며 범위 제한은 저렴한 model이 bash나 filesystem으로
    # 벗어나는 것을 막습니다. security 경계이기도 합니다. worker는
    # 임의의(신뢰하지 않는) web 페이지를 읽으므로 검색하고,
    # fetch하고, 보고만 할 수 있는 worker가 그 입력에 원하는 영향 범위입니다.
    # 그리고 보고서를 읽는 coordinator에는
    # tool이 전혀 없습니다.
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": False},
            "configs": [
                {"name": "web_search", "enabled": True},
                {"name": "web_fetch", "enabled": True},
            ],
        }
    ],
    system=(
        "You are a search worker researching one focused sub-question for "
        "a coordinator. Use web_search and web_fetch to find the answer. "
        "Be thorough: try multiple query phrasings, follow promising "
        "links, and cross-check facts across sources. Report back with "
        "the specific answer you found and the evidence (URLs, quotes) "
        "that supports it. If you could not find a definitive answer, say "
        "exactly what you did find and what remains uncertain. Always "
        "finish by calling submit_result."
    ),
    betas=BETAS,
)

coordinator = client.beta.agents.create(
    name="search-coordinator",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=COORDINATOR_MODEL,
    multiagent={
        "type": "coordinator",
        "agents": [{"type": "agent", "id": worker.id}],
    },
    system=(
        "You are coordinating a team of search workers to answer a hard "
        "web-research question. Your workers have web_search and "
        "web_fetch; you do not. Break the question into focused "
        "sub-questions and delegate each to a worker via create_agent. "
        "Run several workers in parallel on independent sub-questions, "
        "and ALWAYS call wait_for_agents after spawning before drawing "
        "any conclusion. When a worker reports, decide whether its "
        "findings answer the sub-question or whether to send a follow-up "
        "with send_to_agent. If a worker returns an infrastructure error "
        "(rate limit, timeout) instead of findings, re-assign the same "
        "sub-question to a fresh worker. Once you have enough evidence, "
        "synthesize the workers' findings into a single final answer to "
        "the original question."
    ),
    betas=BETAS,
)

print(f"worker      {worker.id}")
print(f"coordinator {coordinator.id}")

## 2. 조사 질문 실행하기

coordinator용 environment와 세션을 생성하고 질문을 `user.message`로 보낸 뒤 streaming합니다. 세션 수준 stream은 coordinator의 primary thread이며 전체 실행의 압축된 보기입니다. worker thread는 위임 traffic으로 나타납니다. worker가 생성될 때 `session.thread_created`, coordinator가 하위 질문을 넘길 때 `agent.thread_message_sent`, 결과가 돌아올 때 `agent.thread_message_received`입니다.

질문은 내용 범위 작업입니다. 사실 스무 개(공원 10개 x 속성 2개)를 각각 지정된 권위 있는 출처로 검증해야 합니다. 내용 범위 질문에서 이 패턴이 빛납니다. 읽기는 필수이기 때문입니다. 누구도 기억만으로 답할 수 없으므로 문제는 읽기에 어떤 요금이 청구되고 병렬로 일어나는지뿐입니다. ([BrowseComp](https://arxiv.org/abs/2504.12516) 같은 benchmark 방식의 발견 질문 — 큰 검색 공간에 숨은 답 하나 찾기 — 은 frontier model의 검색 직관을 더 높이 평가하므로 격차가 좁아집니다.)

작업 분기는 데이터에 따라 달라집니다. coordinator가 worker 수를 결정하므로 prompt는 상한을 고정하지 않습니다. 아래 세션에는 모든 thread에 걸친 전체 팀의 정가 비용에 강제로 적용하는 상한인 `budget`이 있습니다. $10 상한은 정상 실행에서는 도달하지 않을 정도로 충분하지만, 나쁜 질문이 coordinator로 하여금 끝없이 worker를 생성하게 하더라도 청구액을 계속 늘리는 대신 상한에서 세션이 일시 정지됩니다. 일시 정지 방식과 이후 상한을 올리는 방법은 [`CMA_cap_session_spend.ipynb`](CMA_cap_session_spend.ipynb)에 있습니다.


In [ ]:
env = client.beta.environments.create(
    name="research-fanout",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=env.id,
    # 데이터에 따른 작업 분기 제한 장치: 전체 팀의 정가 비용에 상한 설정
    budget={"type": "limit", "max_list_cost": {"currency": "USD", "amount": "1000"}},
    betas=BETAS,
)

QUESTION = (
    "For each of the ten largest national parks in the contiguous United "
    "States by area, find: the current standard private-vehicle entrance "
    "fee, and whether the park currently requires a timed-entry or "
    "day-use reservation for peak season. Each fact must be verified "
    "against that park's official nps.gov pages (fees page and alerts/"
    "reservations page) - not from third-party summaries. Give park, fee, "
    "reservation requirement, and the nps.gov URLs you used."
)

t_start = time.monotonic()

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[{"type": "user.message", "content": [{"type": "text", "text": QUESTION}]}],
)


def text_of(content):
    return "".join(b.text for b in content or [] if b.type == "text")


def clip(s, n=160):
    return s[:n] + ("..." if len(s) > n else "")


final_answer = ""
with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        match ev.type:
            case "agent.message":
                if text := text_of(ev.content).strip():
                    final_answer = text
                    print(f"[coordinator] {clip(text, 200)}")
            case "session.thread_created":
                print(f"[spawn] {ev.agent_name} ({ev.session_thread_id})")
            case "agent.thread_message_sent":
                print(f"[delegate -> {ev.to_agent_name}] {clip(text_of(ev.content))}")
            case "agent.thread_message_received":
                print(f"[report <- {ev.from_agent_name}] {clip(text_of(ev.content))}")
            case "session.status_idle":
                break

print(f"\n[team finished in {time.monotonic() - t_start:.0f}s]")
print("=" * 70)
print(final_answer)

주목할 형태는 모든 `[delegate ->]` 줄이 작은 message이고 모든 `[report <-]` 줄이 정제된 요약이라는 점입니다. 보고서를 만든 수 MB의 검색 결과와 fetch한 페이지는 coordinator context로 넘어오지 않았습니다. 이 분리가 비용 설명의 전부입니다. 공정하게 가격을 비교하기 위해 다음 절에서 현실적인 대안을 실행한 뒤 두 실행을 측정합니다.


## 3. 대조 실행: 같은 검증 기준의 frontier agent 하나

이 패턴이 없으면 비용이 얼마일까요? 현실적인 대안은 같은 두 web tool을 가진 단일 frontier agent입니다. 이 비교를 공정하게 만들거나 무의미하게 만드는 미묘한 점이 하나 있습니다. **단독 agent에도 같은 검증 기준을 적용해야 합니다.** 자체 판단에 맡기면 frontier model은 경제적으로 행동합니다. 사실당 출처 하나를 읽고 저렴하게 끝내지만, 이는 엄밀성이 낮은 결과물이며 같은 작업을 다른 가격으로 수행한 것이 아닙니다. 따라서 아래 단독 prompt는 팀이 이미 하는 것을 요구합니다. 모든 사실을 독립적인 fetch 두 번으로 검증하고 충돌을 다시 확인하여 표시합니다.

같은 질문, 조용한 stream입니다.


In [ ]:
solo = client.beta.agents.create(
    name="solo-researcher",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-plan-big-execute-small"},
    model=COORDINATOR_MODEL,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": False},
            "configs": [
                {"name": "web_search", "enabled": True},
                {"name": "web_fetch", "enabled": True},
            ],
        }
    ],
    system=(
        "You research hard web questions with audit-grade rigor. Use "
        "web_search and web_fetch. For EVERY fact you report, verify it "
        "from at least two independent fetches (the authoritative page "
        "plus one corroborating source), and re-fetch when two sources "
        "disagree. Never carry a fact forward on one source or from "
        "memory. In your answer, give each fact with both source URLs, "
        "and explicitly flag any fact where sources conflicted. Before "
        "finishing, audit your own answer: list each claim and check it "
        "has two cited sources."
    ),
    betas=BETAS,
)

t_solo = time.monotonic()
solo_session = client.beta.sessions.create(agent=solo.id, environment_id=env.id, betas=BETAS)
client.beta.sessions.events.send(
    solo_session.id,
    betas=BETAS,
    events=[{"type": "user.message", "content": [{"type": "text", "text": QUESTION}]}],
)

solo_answer = ""
with client.beta.sessions.events.stream(solo_session.id, betas=BETAS) as stream:
    for ev in stream:
        match ev.type:
            case "agent.message":
                if text := text_of(ev.content).strip():
                    solo_answer = text
            case "session.status_idle":
                break

print(f"[solo finished in {time.monotonic() - t_solo:.0f}s]")
print(clip(solo_answer, 300))

## 4. 두 실행의 사용량과 비용 측정하기

비용 귀속은 API에 내장되어 있습니다. 모든 세션 thread에는 누적 `usage`가 있으며 세션과 각 thread 모두 공개 정가의 token 비용을 서버 측에서 계산한 `usage.list_cost`를 보고합니다. thread를 나열하고 primary thread(`parent_thread_id is None`)를 coordinator로, child thread를 worker로 취급하세요. 단독 세션에는 child thread가 없을 뿐입니다. 세션 자체의 `list_cost`가 합계이므로 주요 수치를 위해 요금표를 동기화할 필요가 없습니다.

(요청별 세부 정보가 필요하다면 각 thread 자체 event feed에 있습니다. 세션 수준 feed에는 primary thread의 `span.model_request_end` event만 있습니다.)

API가 제공할 수 없는 수치는 가정에 따른 비용입니다. 정확히 이 팀의 작업을 모두 frontier 요금으로 청구하면 얼마인가입니다. token 수에 요금표를 적용해야 하므로 코드는 이 가정 하나를 위해 [가격 페이지](https://platform.claude.com/docs/en/about-claude/pricing)의 input 및 output 요금을 보관합니다. 5분 cache write는 input 요금의 1.25배, 1시간 write는 2배, cache read는 0.1배입니다. 실제 값에는 모두 `list_cost`를 사용합니다.


In [ ]:
# 가격 페이지의 input 및 output $ / MTok입니다. 아래의
# 모두 frontier인 가정에만 사용하며 실제 청구액은 API의
# 서버 측 list_cost에서 가져옵니다. Sonnet 5는 도입 요금($2/$10,
# 2026년 8월 31일까지; 이후 표준 $3/$15 — 그때 update)을 표시합니다.
PRICES = {
    "claude-fable-5": {"input": 10.0, "output": 50.0},
    "claude-sonnet-5": {"input": 2.0, "output": 10.0},
}


def total_input(u):
    cache = u.cache_creation  # cache 활동이 없는 thread에서는 None
    return (
        u.input_tokens
        + u.cache_read_input_tokens
        + (cache.ephemeral_5m_input_tokens if cache else 0)
        + (cache.ephemeral_1h_input_tokens if cache else 0)
    )


def counterfactual_cost(u, model):
    """thread의 token 비용을 다른 model 요금으로 다시 계산합니다(가정)."""
    p = PRICES[model]
    cache = u.cache_creation
    return (
        u.input_tokens * p["input"]
        + (cache.ephemeral_5m_input_tokens if cache else 0) * p["input"] * 1.25
        + (cache.ephemeral_1h_input_tokens if cache else 0) * p["input"] * 2.0
        + u.cache_read_input_tokens * p["input"] * 0.1
        + u.output_tokens * p["output"]
    ) / 1e6


def dollars(list_cost):
    return float(list_cost.amount) / 100  # amount는 센트 단위의 정수 문자열입니다


def report(session_id):
    session_usage = client.beta.sessions.retrieve(session_id, betas=BETAS).usage
    threads = list(client.beta.sessions.threads.list(session_id, betas=BETAS))
    primary = next(t for t in threads if t.parent_thread_id is None)
    workers = [t for t in threads if t.parent_thread_id is not None]
    workers_in = sum(total_input(t.usage) for t in workers)
    print(
        f"  primary thread ({primary.agent.model.id}): "
        f"{total_input(primary.usage):>9,} in / {primary.usage.output_tokens:>6,} out"
        f"  -> ${dollars(primary.usage.list_cost):.2f}"
    )
    if workers:
        print(
            f"  {len(workers)} worker(s): {workers_in:>9,} in / "
            f"{sum(t.usage.output_tokens for t in workers):>6,} out"
            f"  -> ${sum(dollars(t.usage.list_cost) for t in workers):.2f}"
        )
        print(
            f"  workers' share of input: {workers_in / (workers_in + total_input(primary.usage)):.0%}"
        )
    total = dollars(session_usage.list_cost)
    print(f"  total cost (session usage.list_cost): ${total:.2f}")
    return total, threads


print("split team (fable coordinator + sonnet workers):")
split_cost, split_threads = report(session.id)

print("\nsolo frontier agent:")
solo_cost, _ = report(solo_session.id)

# 요금 분리 효과만 보는 가정: 이번 실행의 팀 작업에서
# 모든 token을 frontier 요금으로 청구합니다.
frontier_team_cost = sum(counterfactual_cost(t.usage, COORDINATOR_MODEL) for t in split_threads)

print(f"\nsolo / split cost ratio on this pair of runs: {solo_cost / split_cost:.1f}x")
print(f"the split team's workload at all-frontier rates: ${frontier_team_cost:.2f}")

두 실행은 거의 같은 분량을 읽었습니다. 검증 기준을 맞추는 목적입니다. 다른 것은 읽기에 청구된 요금과 작업 형태입니다. 팀의 조회 스무 개는 저렴한 요금의 병렬 worker thread에서 실행되었고, 단독 agent는 frontier 요금의 context 하나에서 차례로 수행했습니다. 작성자의 실행에서는 팀 input token의 84-98%가 worker 요금으로 청구되었습니다. token 양은 실행마다 달라지므로 출력된 비율 하나는 표본 하나로 취급하세요. 안정적인 부분은 구조입니다.

이 notebook을 만들면서 모두 관찰한 네 가지 솔직한 주의점입니다.

* **같은 엄밀성으로 비교하세요.** 자체 판단에 맡긴 단독 frontier agent는 훨씬 적게 읽고(사실당 출처 하나) 팀보다 저렴하지만, 이는 다른, 엄밀성이 낮은 결과물입니다. 검증 기준을 고정한 상태에서만 두 방식을 비교하세요.
* **위임에는 최소 비용이 있습니다.** 각 worker thread는 고정적인 설정 overhead를 지불합니다. 같은 작업을 더 많고 더 좁은 작업 지시로 나누면 비용이 줄지 않고 늘었습니다. 작업 지시의 세분화에는 최적점이 있습니다.
* **검증 기준은 넣은 내용만 다룹니다.** commit된 실행의 두 방식 모두 nps.gov로 사실 스무 개를 검증했지만, 둘 다 model 기억으로 공원 목록을 만들면서 실제로는 Great Smoky Mountains의 자리인 #10에 Kings Canyon을 넣었습니다(Kings Canyon은 면적 기준 #12). 사실은 audit했지만 질문 분해는 하지 않았습니다. 전제가 중요하면 위임을 한 번 더 하여 worker가 검증하게 하세요.
* **coordinator는 알려 준 내용만 압니다.** 서버가 worker prompt를 보여 주지 않으므로(1절 참고) 경제성도 coordinator prompt에서 worker 동작을 정확히 설명하는 데 달려 있습니다.

언제 이 분리가 이득이 *아닐까요*? 좁은 질문에서는 요금 차이를 활용할 읽기 분량이 너무 적습니다. coordinator가 자체 지식으로 답하면(위임 없음) frontier 왕복 비용을 아무 소득 없이 낸 것입니다. `[spawn]` 줄이 없는 실행을 살펴보세요. 또한 작업이 원본 자료 자체에 대한 frontier 판단을 요구하면(사실 찾기 대신 미묘한 문서 분석), 저렴한 reader가 요약하면서 정확히 중요한 내용을 없앨 수 있습니다.

## 되짚어 보기

이 guide에서는 유용한 multi-agent 팀의 가장 저렴한 형태를 만들고 그 이득을 측정했습니다.

* **두 model 팀 설정** — 유일한 기능이 `multiagent` roster인 frontier coordinator와 web tool로 범위를 한정한 저렴한 worker
* 세션 stream의 `thread_created` / `thread_message_sent` / `thread_message_received`를 통해 **위임 실시간으로 따라가기**
* **같은 엄밀성의 단독 frontier 대조 실행**을 수행하고 추정치가 아닌 실제 청구액과 실제 경과 시간 비교
* 모든 세션 thread의 타입이 지정된 누적 `usage`로 **각 thread 측정**(`session.usage`는 전체 팀의 합계)

더 발전시키려면:

1. 범위가 지정된 toolset을 가진 specialist worker 타입을 추가하세요. [`CMA_coordinate_specialist_team.ipynb`](https://github.com/anthropics/claude-cookbooks/blob/main/managed_agents/CMA_coordinate_specialist_team.ipynb)는 세 역할 팀과 역할별 범위 지정의 중요성을 보여 줍니다.
2. thread별 측정을 production telemetry에 넣으세요. thread 수준 usage가 세션별뿐 아니라 위임별 지출을 귀속하는 방법입니다.
3. token을 많이 사용하는 자체 작업에도 같은 분리를 적용해 보세요. 문서 review와 log triage는 web 조사처럼 읽기가 많고 내용 범위를 다루는 성격을 가집니다.

참고: [multi-agent 세션 문서](https://platform.claude.com/docs/en/managed-agents/multi-agent).

